# Exercise 2. Build a Chatbot
In this exercise, we'll spend some time building a simple chatbot :)

## 2.1 Setup
Let's start by importing what we need (should have been installed during previous exercise!)

In [ ]:
from transformers import AutoTokenizer, pipeline

## 2.2 Define a Chatbot Class
When should you use a class in Python? This can be tricky to decide. As a rule of thumb, if you do not need to *store* and *carry* information between operations, functions are usually enough.

**A chatbot is a good example of when a class is useful**, as it needs to hold information across multiple interactions. For example, a `chat_history` containing all user and LLM messages gives the chatbot *a form of memory*. See the overview below:

```{figure} ../figures/class6/chatbot-class-components.png
---
name: zero-shot-few-shot-fine-tuning overview
width: 100%
---
AI-generated, modified by me. Note that an *api_key* is only relevant for models hosted on other inference servers (e.g,. from third-party GPU cloud platforms or using OpenAI's models).
```

Let's define our *chatbot* class. Some of this may look a little new to some of you, but we'll try to break things down and slowly increase in complexity:

In [ ]:
class Chatbot:
    """
    Simple chatbot with a chat history. 

    Defaults to Qwen/Qwen3-0.6B if no model_id is provided.
    -------------

    """
    def __init__(self, model_id: str = "Qwen/Qwen3-0.6B"):
        self.model_id = model_id
        self.chat_history = []

    def load(self):
        pass

Let us look at `__init__()`. This special method is called when we create an instance of the class (for example, `chatbot = Chatbot(...)`) and initializes all attributes. Some attributes are provided by the user, while *internal attributes* are not!

```{figure} ../figures/class6/class_init.jpg
---
name: class_init
width: 100%
---
By me (using [carbon.sh](https://carbon.now.sh/?bg=rgba(255,255,255,0)&t=one-light&wt=none&l=python&width=824&ds=false&dsyoff=20px&dsblur=68px&wc=true&wa=false&pv=56px&ph=56px&ln=false&fl=1&fm=Hack&fs=14px&lh=133%25&es=2x&wm=false&code=class%2520Chatbot%253A%250A%2520%2520%2520%2520def%2520__init__(self%252C%2520model_id)%253A%250A%2520%2520%2520%2520%2520%2520%2520%2520self.model_id%2520%253D%2520model_id%250A%2520%2520%2520%2520%2520%2520%2520%2520self.chat_history%2520%253D%2520%255B%255D))
```

Let's add `self.model` and `self.tokenizer` and set them to `None` for now (we'll load them later!)

In [ ]:
class Chatbot:
    """
    Simple chatbot with a chat history. 

    Defaults to Qwen/Qwen3-0.6B if no model_id is provided.
    -------------

    """
    def __init__(self, model_id: str = "Qwen/Qwen3-0.6B"):
        self.model_id = model_id
        self.chat_history = []
        self.tokenizer = None
        self.model = None

    def load(self):
        pass

### Your Turn: Add Temperature as a Parameter
:::{admonition} HANDS-ON
:class: red
As a simple addition, add a `temperature` parameter that the user can pass to `Chatbot()` (to later be used in `pipeline`). Make it default to `None`, so that a user *can* but is not *forced* to specify it.

Optionally, you can specify [type hint](https://docs.python.org/3/library/typing.html) that corresponds to type of value that `temperature` can take (see [docs](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation#transformers.GenerationConfig.temperature))
:::

#### Solution

In [ ]:
class Chatbot:
    """
    Simple chatbot with a chat history. 

    Defaults to Qwen/Qwen3-0.6B if no model_id is provided.
    -------------

    """
    def __init__(self, model_id: str = "Qwen/Qwen3-0.6B", temperature: float = None):
        self.model_id = model_id
        self.chat_history = []
        self.tokenizer = None
        self.model = None
        self.temperature = temperature

    def load(self):
        pass

## 2.3 Loading an LLM with our Chatbot
As you may have noticed, our Chatbot class has an unspecified `load` method. Let's define what that this function should do (and how it should use our attributes). 

### Your Turn: Define a Load Method
:::{admonition} HANDS-ON
:class: red
Remove `pass` from `def load(self)` and start filling out the function!
1.  The load function should load the tokenizer and `pipeline` as we did in the previous exercise!
2. Add `task` as a parameter in load with the default `task = text_generation`
:::

If you are unfamiliar with class attributes, I suggest you read the little hint box before proceeding!

:::{admonition} Using attributes in methods?
:class: tip, dropdown
Imagine a Person class like this:
```python
class Person:
   def __init__(self, name, favorite_color):
        self.name = name
        self.favorite_color = favorite_color
```

When we intialise a Person class, we can access its attributes:
```python
mina = Person(name = "Mina", favorite_color = "green")
print(mina.name) # prints Mina
```

This is great! But where attributes really shine is how they are used in subsequent methods! Let's say we want an "introduction" method:
```python
class Person:
   def __init__(self, name, favorite_color):
        self.name = name
        self.favorite_color = favorite_color

   def introduction(self):
        intro = f"My name is {self.name} and my favorite color is {self.favorite_color}."
        print(intro)
```
Note that in the "introduction" method above, we aren't passing any parameters other than `self` (the class instance itself). But since `self` has attributes like `name` and `color`, we can make our intro! This would look like this:
```python
mina = Person(name = "Mina", favorite_color = "green")
mina.introduction()  # prints "My name is Mina and my favorite color is green."
```

We *can* also add parameters to specific methods! Let's say we want to *optionally* have a special way of saying "goodbye"
```python
class Person:
   def __init__(self, name, favorite_color):
        self.name = name
        self.favorite_color = favorite_color

     def introduction(self, special_goodbye=None):
          intro = f"My name is {self.name} and my favorite color is {self.favorite_color}."
          
          if special_goodbye: # add extra to outro
               intro += f" {special_goodbye}"
          
          print(intro)
```

When using the `introduction` method, we can now optionally define our special outro:
```python
mina.introduction(special_goodbye="Peace out!")  # prints "My name is Mina and my favorite color is Green. Peace out!"
```
:::


#### Solution

In [11]:
class Chatbot:
    """
    Simple chatbot with a chat history. 

    Defaults to Qwen/Qwen3-0.6B if no model_id is provided.
    -------------
    """
    def __init__(self, model_id: str = "Qwen/Qwen3-0.6B", temperature: float = None):
        self.model_id = model_id
        self.chat_history = []
        self.tokenizer = None
        self.model = None
        self.temperature = temperature

    def load(self, task = "text-generation"):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        
        self.model = pipeline(
            task=task,
            model=self.model_id,
            tokenizer=self.tokenizer,
            temperature=self.temperature
        )

### Lazy Loading
After having created your `load` function, consider firstly: 

:::{admonition} QUESTION
:class: red
Why do we define `task` as a parameter in `load` and not as a class attribute?

<details>
<summary>ANSWER</summary>
Firstly, you could argue that <code>task</code> is not an intrinsic property of the Chatbot class like <code>model_id</code> is. 

Secondly ...

</details>
:::

Now, let's make our `load` function a bit smarter:
:::{admonition} HANDS-ON
:class: red
Let's make our `load` function a **lazy loader**:
- Add conditions to both the loading of `tokenizer` and `pipeline`, so that the lines are *only* run if `self.tokenizer` and `self.model` is None.
:::